# 02 — Prompt + LangGraph agent

Spojíme tři dílky:
1. **System prompt** pullnutý z LangSmith Hubu (tag podle `ENVIRONMENT`).
2. **Tools** z `kosik_workshop.tools` (`ALL_TOOLS`).
3. **LangGraph** ReAct loop s explicitními nody `call_model` a `tools`, podmíněná hrana podle `tool_calls`.

Cíl: vidět celý kruh — prompt rozhoduje o tool callech, tool výsledky jdou zpátky do modelu, finální odpověď v češtině.

In [1]:
from kosik_workshop.config import load_env
load_env()

## 1. Pull promptu z LangSmith

Loader respektuje `ENVIRONMENT` (default `development` → tag `dev`) a optional `KOSIK_PROMPT_COMMIT` pin.

In [2]:
from kosik_workshop.prompts.loader import load_kosik_prompt
prompt = load_kosik_prompt()
system_text = next(
    msg.prompt.template for msg in prompt.messages
    if getattr(msg, 'prompt', None) and getattr(msg.prompt, 'template', None)
)
print(system_text[:500], '...')

Jsi nákupní asistent e-shopu Košík.cz. Pomáháš zákazníkům najít produkty, porovnat varianty a zvládnout nákup s ohledem na jejich preference a alergeny.

## Co umíš
- Vyhledávat produkty v katalogu podle přirozeného dotazu, kategorie, ceny nebo požadavku na veganskou variantu.
- Zjišťovat detailní informace o konkrétním produktu (složení, alergeny, cena, původ).
- Kontrolovat, zda je produkt bezpečný vzhledem k alergenům uživatele.
- Zjistit aktuální seznam alergenů přihlášeného uživatele.

Nást ...


## 2. Sestavení grafu

`build_agent()` postaví `StateGraph` s nody:
- **`call_model`**: prepend system message k historii, zavolá LLM s bind_tools.
- **`tools`**: prebuilt `ToolNode` — exekuuje `tool_calls` z poslední AI zprávy.

Conditional edge: pokud poslední zpráva obsahuje `tool_calls` → jdi do `tools`, jinak END.

Graf má napojený `MemorySaver` checkpointer — invokace se stejným `thread_id` sdílí konverzační stav.

In [3]:
from kosik_workshop.agent import build_agent
agent = build_agent()
print('Nodes:', list(agent.nodes.keys()))

Nodes: ['__start__', 'call_model', 'tools']


In [4]:
# Vizualizace grafu (Mermaid → PNG přes pyppeteer/online API).
from IPython.display import Image
try:
    Image(agent.get_graph().draw_mermaid_png())
except Exception as e:
    print('Vizualizace selhala (offline?):', e)
    print(agent.get_graph().draw_mermaid())

## 3. Konverzační session

Vygenerujeme jedno `thread_id` pro celou sezení. Každý `ask()` ho předá do `config.configurable.thread_id`:
- LangGraph checkpointer pod tímhle ID drží stav → druhý turn vidí historii prvního.
- LangSmith UI grupuje runs se stejným `thread_id` do jednoho thread view → vidíš celou konverzaci jako navazující sekvenci.

Reset konverzace = re-run téhle buňky (vygeneruje nové ID, MemorySaver pod tím ID nemá nic).

In [ ]:
import uuid
session_thread_id = str(uuid.uuid4())
# V notebooku user_id hardcoduje. V API ho doplní auth middleware z access tokenu.
current_user_id = 'demo-user-1'
print('thread_id:', session_thread_id)
print('user_id:  ', current_user_id)

In [ ]:
from langchain_core.messages import HumanMessage

def ask(question: str, feature: str | None = None, user_id: str | None = None) -> None:
    print(f'\n=== {question} ===')
    tags = ['surface:notebook']
    if feature:
        tags.append(f'feature:{feature}')
    metadata = {
        'thread_id': session_thread_id,   # LangGraph checkpointer key
        'session_id': session_thread_id,  # LangSmith Threads view groups by session_id
    }
    if user_id := (user_id or current_user_id):
        metadata['user_id'] = user_id
    config = {
        'configurable': {'thread_id': session_thread_id},
        'metadata': metadata,
        'tags': tags,
    }
    result = agent.invoke({'messages': [HumanMessage(content=question)]}, config=config)
    new_messages = [m for m in result['messages'] if m.type != 'human'][-10:]
    for msg in new_messages:
        if msg.type == 'ai' and getattr(msg, 'tool_calls', None):
            for tc in msg.tool_calls:
                print(f'  → tool: {tc["name"]}({tc["args"]})')
    print('\nOdpověď:')
    print(result['messages'][-1].content)

## 4. Konverzace s navazujícími dotazy

Druhý a třetí dotaz záměrně odkazují na předchozí kontext ("a co kdyby…", "doporučte mi něco podobného") — bez sdíleného threadu by model nevěděl, na co se odkazuje.

In [7]:
ask('Hledám veganské mléko do 50 Kč.', feature='product-search')


=== Hledám veganské mléko do 50 Kč. ===
  → tool: search_products({'query': 'mléko', 'max_price_czk': 50, 'vegan_only': True})

Odpověď:
Bohužel jsem nenašel žádné veganské mléko do 50 Kč. Mohu vám nabídnout alternativní produkty, které jsou veganské, ale nejsou mléko. Například:

- **Mlýn Pernštejn Celozrnná mouka 1 kg** (`mlyn-pernstejn-celozrnna-mouka-1-kg`)
  - Cena: 30 Kč / kg
  - Důvod: Celozrnná mouka je zdravou alternativou pro pečení.

- **Mlýn Pernštejn Hladká mouka 1 kg** (`mlyn-pernstejn-hladka-mouka-1-kg`)
  - Cena: 22 Kč / kg
  - Důvod: Hladká mouka je univerzální a vhodná pro různé recepty.

- **Odkolek Chléb žitný 400 g** (`odkolek-chleb-zitny-400-g`)
  - Cena: 45 Kč / g
  - Důvod: Žitný chléb je chutný a veganský.

- **Mlýn Pernštejn Pšeničná mouka 1 kg** (`mlyn-pernstejn-psenicna-mouka-1-kg`)
  - Cena: 29 Kč / kg
  - Důvod: Pšeničná mouka je ideální pro pečení.

- **Mrkev baby 500 g** (`mrkev-baby-500-g`)
  - Cena: 29 Kč / balení
  - Důvod: Čerstvá zelenina je vždy z

In [8]:
ask('A co kdybych připustil cenu do 80 Kč?', feature='product-search')


=== A co kdybych připustil cenu do 80 Kč? ===
  → tool: search_products({'query': 'mléko', 'max_price_czk': 50, 'vegan_only': True})
  → tool: search_products({'query': 'mléko', 'max_price_czk': 80, 'vegan_only': True})

Odpověď:
Bohužel ani při zvýšení cenového limitu na 80 Kč jsem nenašel žádné veganské mléko. Mohu vám však nabídnout další veganské produkty, které jsou v této cenové kategorii:

- **Mlýn Pernštejn Celozrnná mouka 1 kg** (`mlyn-pernstejn-celozrnna-mouka-1-kg`)
  - Cena: 30 Kč / kg
  - Důvod: Celozrnná mouka je zdravou alternativou pro pečení.

- **Mlýn Pernštejn Hladká mouka 1 kg** (`mlyn-pernstejn-hladka-mouka-1-kg`)
  - Cena: 22 Kč / kg
  - Důvod: Hladká mouka je univerzální a vhodná pro různé recepty.

- **Odkolek Chléb žitný 400 g** (`odkolek-chleb-zitny-400-g`)
  - Cena: 45 Kč / g
  - Důvod: Žitný chléb je chutný a veganský.

- **Mlýn Pernštejn Pšeničná mouka 1 kg** (`mlyn-pernstejn-psenicna-mouka-1-kg`)
  - Cena: 29 Kč / kg
  - Důvod: Pšeničná mouka je ideální p

In [9]:
ask('Mám alergii na lepek. Které z těch produktů jsou pro mě bezpečné?', feature='allergen-check')


=== Mám alergii na lepek. Které z těch produktů jsou pro mě bezpečné? ===
  → tool: search_products({'query': 'mléko', 'max_price_czk': 80, 'vegan_only': True})
  → tool: check_allergens({'product_id': 'mlyn-pernstejn-celozrnna-mouka-1-kg', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'mlyn-pernstejn-hladka-mouka-1-kg', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'odkolek-chleb-zitny-400-g', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'mlyn-pernstejn-psenicna-mouka-1-kg', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'odkolek-chleb-se-seminky-500-g', 'user_allergens': ['lepek']})

Odpověď:
Bohužel všechny uvedené produkty obsahují lepek, což znamená, že pro vás nejsou bezpečné. Zde je shrnutí:

- **Mlýn Pernštejn Celozrnná mouka 1 kg** — **Obsahuje lepek**.
- **Mlýn Pernštejn Hladká mouka 1 kg** — **Obsahuje lepek**.
- **Odkolek Chléb žitný 400 g** — **Obsahuje lepek**.
- **Mlýn Pe

In [10]:
ask('Jaké alergeny mám v profilu?', feature='profile-query')


=== Jaké alergeny mám v profilu? ===
  → tool: check_allergens({'product_id': 'mlyn-pernstejn-celozrnna-mouka-1-kg', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'mlyn-pernstejn-hladka-mouka-1-kg', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'odkolek-chleb-zitny-400-g', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'mlyn-pernstejn-psenicna-mouka-1-kg', 'user_allergens': ['lepek']})
  → tool: check_allergens({'product_id': 'odkolek-chleb-se-seminky-500-g', 'user_allergens': ['lepek']})
  → tool: user_allergens({})

Odpověď:
Ve vašem profilu nemáte žádné alergeny. Pokud byste chtěl hledat produkty bez specifických alergenů, dejte mi vědět, a já vám rád pomohu!


In [11]:
ask('Máte v nabídce čerstvého kapra?', feature='product-search')


=== Máte v nabídce čerstvého kapra? ===
  → tool: user_allergens({})
  → tool: search_products({'query': 'kapr', 'category': 'Čerstvé ryby'})

Odpověď:
Bohužel jsem nenašel žádného čerstvého kapra v nabídce. Pokud máte zájem o jiný typ ryby nebo jinou kategorii produktů, dejte mi vědět a rád vám pomohu!


## 5. Inspekce stavu

Checkpointer drží snapshot stavu pod `thread_id`. Můžeš se podívat, co všechno LangGraph nakumuloval.

In [12]:
snapshot = agent.get_state({'configurable': {'thread_id': session_thread_id}})
print(f'Zpráv ve stavu: {len(snapshot.values["messages"])}')
for m in snapshot.values['messages']:
    preview = (m.content if isinstance(m.content, str) else str(m.content))[:80].replace('\n', ' ')
    print(f'  [{m.type:5s}] {preview}')

Zpráv ve stavu: 24
  [human] Hledám veganské mléko do 50 Kč.
  [ai   ] 
  [tool ] [{"id": "mlyn-pernstejn-celozrnna-mouka-1-kg", "name": "Mlýn Pernštejn Celozrnná
  [ai   ] Bohužel jsem nenašel žádné veganské mléko do 50 Kč. Mohu vám nabídnout alternati
  [human] A co kdybych připustil cenu do 80 Kč?
  [ai   ] 
  [tool ] [{"id": "mlyn-pernstejn-celozrnna-mouka-1-kg", "name": "Mlýn Pernštejn Celozrnná
  [ai   ] Bohužel ani při zvýšení cenového limitu na 80 Kč jsem nenašel žádné veganské mlé
  [human] Mám alergii na lepek. Které z těch produktů jsou pro mě bezpečné?
  [ai   ] 
  [tool ] {"safe": false, "conflicts": ["lepek"], "product_allergens": ["lepek"], "user_al
  [tool ] {"safe": false, "conflicts": ["lepek"], "product_allergens": ["lepek"], "user_al
  [tool ] {"safe": false, "conflicts": ["lepek"], "product_allergens": ["lepek"], "user_al
  [tool ] {"safe": false, "conflicts": ["lepek"], "product_allergens": ["lepek"], "user_al
  [tool ] {"safe": false, "conflicts": ["lepek"], "pro

## 6. LangSmith trace

Otevři https://eu.smith.langchain.com → projekt `kosik-workshop` → záložka **Threads**. 
Najdi thread podle `session_thread_id` (vytištěný výše) — uvidíš všechny `ask()` runy seskupené pod jednou konverzací, navazující turn-by-turn.

V každém runu uvidíš:
- pullnutý prompt commit (z metadat),
- LLM call s tool definitions,
- `ToolNode` exekuci s argumenty a výsledkem,
- finální AI message.

Iterace: uprav prompt v Playgroundu → commit s tagem `dev` → re-run notebook (`load_kosik_prompt` pullne novou verzi).

In [ ]:
from kosik_workshop.tracing import scrub_text

samples = [
    'Můj email je info@malikpetr.cz, zavolej mi na +420 777 123 456.',
    'IBAN CZ65 0800 0000 1920 0014 5399, karta 4111 1111 1111 1111.',
    'Rodné číslo 901231/1234.',
]
for s in samples:
    print(s)
    print(' →', scrub_text(s))
    print()

## 7. PII redaction (GDPR)

Tři vrstvy maskování dat poslaných do LangSmith:

1. **Pre-send scrubber** (`src/kosik_workshop/tracing/redaction.py`) — regex-based redaction emailů, telefonů, IBAN, čísel karet, rodných čísel. Spouští se v procesu před odesláním.
2. **Full hide** (`LANGSMITH_HIDE_INPUTS=true` v `.env`) — paranoidní mód, traces budou prázdné.
3. **Server-side redaction rules** — v LangSmith UI: Project Settings → Rules → regex masking.

Pro selektivní scrubování postav `Client` přes `make_redacting_client()` a předej ho LangChain tracingu (typicky přes ENV nebo callback handler).